In [ ]:
# Import

import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import widgets
from ipyfilechooser import FileChooser
from IPython.display import display, clear_output

sys.path.append(os.path.abspath('..'))
from processing.create_paired_dataset_mask import build_paired_roi_stacks_with_mask

In [ ]:
# build ROIs

start_dir = os.path.abspath('../../data') if os.path.isdir('../../data') else os.getcwd()

fc_high = FileChooser(start_dir, title='<b>High SNR image, aligned (*_shifted.tif)</b>')
fc_high.filter_pattern = ['*.tif', '*.tiff', '*.stk']

fc_low = FileChooser(start_dir, title='<b>Low SNR image, raw (.tif)</b>')
fc_low.filter_pattern = ['*.tif', '*.tiff', '*.stk']

# Spatial parameters
w_size_ROI_fit = widgets.IntText(value=8, description='Size of the ROI for fitting:', style={'description_width': 'initial'})
w_threshold = widgets.FloatText(value=25.0, description='Threshold for fitting:', style={'description_width': 'initial'})
w_size_ROI_crop = widgets.IntText(value=64, description='ROI Patch size:', style={'description_width': 'initial'})
w_border_margin = widgets.IntText(value=5, description='Border Margin:', style={'description_width': 'initial'})
w_t_window_low = widgets.IntText(value=3, description='Time window Low:', style={'description_width': 'initial'})

# Mask parameters
w_mask_size = widgets.IntText(value=7, description='Mask size (square, px):', style={'description_width': 'initial'})
w_threshold_mask = widgets.FloatText(value=25.0, description='Threshold for the mask:', style={'description_width': 'initial'})

# Sampling strategy
w_n_patches = widgets.IntText(value=2000, description='Number of training patches:', style={'description_width': 'initial'})
w_ratio_loc = widgets.FloatSlider(value=0.75, min=0.0, max=1.0, step=0.05, description='Loc Ratio:', style={'description_width': 'initial'})
w_output_dir = widgets.Text(value='Training_L5', description='Output folder name:', style={'description_width': 'initial'})

btn_run = widgets.Button(description="Build ROI Stacks + masks", button_style='success', icon='cogs', layout=widgets.Layout(width='auto', height='40px'))
out_logs = widgets.Output()

ui = widgets.VBox([
    widgets.HTML("<h3>Files Loading</h3>"),
    widgets.HBox([fc_high, fc_low]),

    widgets.HTML("<hr><h3>Temporal Window & Localization Process</h3>"),
    widgets.VBox([w_size_ROI_fit, w_threshold, w_size_ROI_crop, w_border_margin, w_t_window_low]),

    widgets.HTML("<hr><h3>Localization mask (L5)</h3>"),
    widgets.VBox([w_mask_size, w_threshold_mask]),

    widgets.HTML("<hr><h3>Training dataset parameters</h3>"),
    widgets.HBox([w_n_patches, w_ratio_loc]),
    w_output_dir,

    widgets.HTML("<br>"),
    btn_run,
    out_logs
])

def on_run_clicked(b):
    global stack_high, stack_low, stack_mask
    with out_logs:
        clear_output(wait=True)

        if not fc_high.selected or not fc_low.selected:
            print("Error: You must select both a High SNR image and a Low SNR image.")
            return

        if w_t_window_low.value % 2 == 0:
            print("Error: The temporal window must be an odd integer.")
            return

        # Output folder is resolved relative to the HIGH image directory, unless absolute
        output_dir = w_output_dir.value
        if not os.path.isabs(output_dir):
            output_dir = os.path.join(os.path.dirname(fc_high.selected), output_dir)

        print(f"Saving in {output_dir}/High, /Low and /Mask (the previous patches are replaced)")
        t0 = time.time()

        try:
            stack_high, stack_low, stack_mask = build_paired_roi_stacks_with_mask(
                [fc_high.selected], [fc_low.selected],
                n_patches=w_n_patches.value,
                t_window_high=1,
                t_window_low=w_t_window_low.value,
                threshold=w_threshold.value,
                size_ROI_fit=w_size_ROI_fit.value,
                size_ROI_crop=w_size_ROI_crop.value,
                border_margin=w_border_margin.value,
                seed=42,
                output_dir=output_dir,
                output_prefix="patches",
                ratio_loc=w_ratio_loc.value,
                mask_size=w_mask_size.value,
                threshold_mask=w_threshold_mask.value
            )
            print(f"Finished in {time.time() - t0:.2f} second(s).")

        except Exception as e:
            print(f"Pipeline Execution Error: {e}")

btn_run.on_click(on_run_clicked)
display(ui)

In [ ]:
# Visualization

idx = np.random.default_rng().choice(len(stack_high), 8, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for a, i in zip(axes.ravel(), idx):
    a.imshow(stack_high[i, 0], cmap='magma')
    a.contour(stack_mask[i, 0] == 1, levels=[0.5], colors='cyan', linewidths=1)      # molecules at t
    a.contour(stack_mask[i, 0] == 2, levels=[0.5], colors='orange', linewidths=1)    # ghost sites    a.set_title(f'patch {i}')
    a.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from scipy import ndimage as ndi
MASK_SIZE = 7
offs = []
for h, m in zip(stack_high[:, 0].astype(float), stack_mask[:, 0]):
    lab, n = ndi.label(m)
    for k in range(1, n + 1):
        rr, cc = np.nonzero(lab == k)
        if len(rr) != MASK_SIZE ** 2:          # only full, isolated squares
            continue
        w = np.clip(h[rr, cc] - np.median(h), 0, None)
        if w.sum() > 0:
            offs.append(((w * rr).sum() / w.sum() - rr.mean(), (w * cc).sum() / w.sum() - cc.mean()))
offs = np.array(offs)
print(len(offs), "squares, mean offset of the spot in its square (rows, cols):", offs.mean(0).round(2),
      "+/-", (offs.std(0) / np.sqrt(len(offs))).round(2))

In [ ]:
import tifffile
from scipy import ndimage as ndi

def wavelet_plane2(img):
    """Second plane of the 'a trous' wavelet transform (B3 spline), the usual SMLM detection filter."""
    k1 = np.array([1, 4, 6, 4, 1]) / 16
    k2 = np.array([1, 0, 4, 0, 6, 0, 4, 0, 1]) / 16
    a1 = ndi.convolve1d(ndi.convolve1d(img, k1, axis=0, mode='mirror'), k1, axis=1, mode='mirror')
    a2 = ndi.convolve1d(ndi.convolve1d(a1, k2, axis=0, mode='mirror'), k2, axis=1, mode='mirror')
    return a1 - a2

def wavelet_noise(path, n_frames=50):
    """Robust noise (MAD) of the wavelet plane, on n_frames frames spread over the movie."""
    movie = tifffile.imread(path)
    frames = movie[np.linspace(0, len(movie) - 1, n_frames).astype(int)].astype(float)
    w = np.stack([wavelet_plane2(f) for f in frames])
    return 1.4826 * np.median(np.abs(w - np.median(w)))

s_high = wavelet_noise(fc_high.selected)
s_low = wavelet_noise(fc_low.selected)
print(f"HIGH: wavelet noise = {s_high:.1f} ADU -> threshold 200 = {200 / s_high:.1f} x noise  (generator threshold 25 = {25 / s_high:.1f} x noise)")
print(f"LOW : wavelet noise = {s_low:.1f} ADU -> threshold 60 = {60 / s_low:.1f} x noise")

In [ ]:
def detect_w2(img, thr):
    """Local maxima of the wavelet plane above thr (our version of the PALMTracer detection)."""
    w = wavelet_plane2(img.astype(float))
    return (w == ndi.maximum_filter(w, 3)) & (w > thr)

THR = 200
found = in_sq = n_sq = sq_hit = 0
for h, m in zip(stack_high[:, 0], stack_mask[:, 0]):
    peaks = detect_w2(h, THR)
    found += peaks.sum()
    in_sq += (peaks & (m > 0)).sum()
    lab, n = ndi.label(m)
    n_sq += n
    sq_hit += len(set(np.unique(lab[peaks])) - {0})
print(f"Our detections: {found}, {100 * in_sq / found:.0f} % inside a PALMTracer square")
print(f"PALMTracer squares (touching squares count as one): {n_sq}, {100 * sq_hit / n_sq:.0f} % contain one of our detections")